# Map and Recude exploration

In [1]:
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.text_splitter import CharacterTextSplitter

In [2]:
# Map prompt: Summarize each chunk
map_template = """The following is a chunk of text:
{text}
Provide a concise summary of this chunk:"""
map_prompt = PromptTemplate.from_template(map_template)

# Reduce prompt: Combine all summaries into one
reduce_template = """The following are summaries of different chunks of text:
{text}
Combine these summaries into one coherent final summary:"""
reduce_prompt = PromptTemplate.from_template(reduce_template)

In [3]:
from langchain_google_vertexai import (
    ChatVertexAI,
    HarmBlockThreshold,
    HarmCategory,
)

safety_settings = {
    HarmCategory.HARM_CATEGORY_UNSPECIFIED: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
}

llm = ChatVertexAI(model="gemini-1.5-pro", temperature=0, safety_settings=safety_settings)

In [ ]:
MAX_TOEKNS = 128000

In [4]:
# Map chain: Summarize each chunk
map_chain = LLMChain(llm=llm, prompt=map_prompt)

# Reduce chain: Combine summaries
reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)

# Combine documents using StuffDocumentsChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain,
    document_variable_name="text"
)

# Reduce documents chain
reduce_documents_chain = ReduceDocumentsChain(
    combine_documents_chain=combine_documents_chain,
    collapse_documents_chain=combine_documents_chain,
    token_max=MAX_TOEKNS  # Adjust based on model token limits
)

# Map-reduce chain
map_reduce_chain = MapReduceDocumentsChain(
    llm_chain=map_chain,
    reduce_documents_chain=reduce_documents_chain,
    document_variable_name="text"
)

/tmp/ipykernel_326651/55938106.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  map_chain = LLMChain(llm=llm, prompt=map_prompt)
/tmp/ipykernel_326651/55938106.py:8: LangChainDeprecationWarning: This class is deprecated. Use the `create_stuff_documents_chain` constructor instead. See migration guide here: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain/
  combine_documents_chain = StuffDocumentsChain(
/tmp/ipykernel_326651/55938106.py:14: LangChainDeprecationWarning: This class is deprecated. Please see the migration guide here for a recommended replacement: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain/
  reduce_documents_chain = ReduceDocumentsChain(
/tmp/ipykernel_326651/55938106.py:21: LangChainDeprecationWarning: This class is deprecated. Please see the migration guide here for a recommended

In [5]:
text_splitter = CharacterTextSplitter(
    chunk_size=MAX_TOEKNS/5,  # Adjust based on your needs
    chunk_overlap=200  # Overlap to maintain context
)

In [6]:

# Example text
text = "Your long input text goes here..."

# Split the text
docs = text_splitter.create_documents([text])

In [7]:
summary = map_reduce_chain.run(docs)
print("Final Summary:", summary)

/tmp/ipykernel_326651/1865348454.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  summary = map_reduce_chain.run(docs)


Final Summary: Please provide me with the summaries you want me to combine. I need the actual summaries to create a coherent final summary for you. 😊 

For example, you could say:

"Here are the summaries:
* **Summary 1:** This article discusses the benefits of regular exercise for physical and mental health.
* **Summary 2:** This section explains the different types of exercise and their specific advantages.
* **Summary 3:** The author concludes by emphasizing the importance of finding an enjoyable exercise routine." 

Once you provide the summaries, I can combine them into a single, comprehensive summary. 👍 



In [19]:
from summarization import summarize_long_text

long_text = """
Patient John Doe visited City Hospital on 2023-10-01 for a routine check-up. Dr. Smith, a cardiologist, conducted the examination. The total cost was $200, covered by insurance. The patient was prescribed medication for high blood pressure and advised to return in 3 months.
    """

In [20]:
summary = summarize_long_text(long_text)
print("Final Summary:\n", summary)

Final Summary:
 ## Patient Summary: John Doe

**Date:** 2023-10-01

**Patient:** John Doe 

**Healthcare Provider:** Dr. Smith, Cardiologist at City Hospital

**Reason for Visit:** Routine check-up

**Medical Summary:**  John Doe underwent a routine check-up with Dr. Smith. He was prescribed medication for high blood pressure and advised to schedule a follow-up appointment in 3 months. 

**Financial Summary:** The total cost of the visit was $200, which was covered by the patient's insurance. 

